# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 6 — Final Results, Learning Curve, and Pipeline Bundle

Generates learning curves, final classification report, publication-quality research plots, and saves the complete deployment pipeline bundle.

### Setup

In [18]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


In [20]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [21]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


In [22]:
def engineer_features(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["CompositeRiskScore"]  = (d["HighBP"] + d["HighChol"] +
                                 d["Smoker"] + d["Stroke"] +
                                 d["HeartDiseaseorAttack"] + d["DiffWalk"])
    d["BMI_Underweight"]     = (d["BMI"] < 18.5).astype(int)
    d["BMI_Normal"]          = ((d["BMI"] >= 18.5) & (d["BMI"] < 25)).astype(int)
    d["BMI_Overweight"]      = ((d["BMI"] >= 25) & (d["BMI"] < 30)).astype(int)
    d["BMI_Obese"]           = (d["BMI"] >= 30).astype(int)
    d["BMI_SevereObese"]     = (d["BMI"] >= 35).astype(int)
    d["BMI_x_PhysActivity"]  = d["BMI"] * d["PhysActivity"]
    d["Age_x_BMI"]           = d["Age"] * d["BMI"]
    d["GenHlth_x_BMI"]       = d["GenHlth"] * d["BMI"]
    d["CardioRisk"]          = (d["HighBP"] * d["HighChol"] +
                                 d["HeartDiseaseorAttack"] + d["Stroke"])
    d["LifestyleScore"]      = (d["PhysActivity"] + d["Fruits"] +
                                 d["Veggies"] - d["Smoker"] -
                                 d["HvyAlcoholConsump"])
    d["HealthAccess"]        = d["AnyHealthcare"] - d["NoDocbcCost"]
    d["HealthBurden"]        = (d["MentHlth"] + d["PhysHlth"]) / 60.0
    d["GenHlth_Sq"]          = d["GenHlth"] ** 2
    d["BMI_Sq"]              = d["BMI"] ** 2
    d["Age_Sq"]              = d["Age"] ** 2
    d["SocioEconomic"]       = d["Income"] * d["Education"]
    d["AgeGroup_Senior"]     = (d["Age"] >= 9).astype(int)
    d["AgeGroup_Middle"]     = ((d["Age"] >= 5) & (d["Age"] < 9)).astype(int)
    d["AgeGroup_Young"]      = (d["Age"] < 5).astype(int)
    d["HighRiskFlag"]        = ((d["HighBP"] == 1) &
                                 (d["HighChol"] == 1) &
                                 (d["BMI_Obese"] == 1)).astype(int)
    d["AlcSmoke"]            = d["HvyAlcoholConsump"] + d["Smoker"]
    d["UncheckedRisk"]       = ((d["CholCheck"] == 0) &
                                 (d["CompositeRiskScore"] > 2)).astype(int)
    d["MentalPhysRatio"]     = (d["MentHlth"] / (d["PhysHlth"] + 1))
    return d
print("✅ engineer_features defined.")

✅ engineer_features defined.


### Load Outputs from Notebook 5

In [23]:
TARGET            = "Diabetes_binary"
FEATURES_ORIGINAL = load_pkl(DIRS["reports"]    / "features_original.pkl")
top_features      = load_pkl(DIRS["features"]   / "top_features.pkl")
tuned_model       = load_pkl(DIRS["models"]     / "tuned_model.pkl")
tuned_metrics     = load_pkl(DIRS["reports"]    / "tuned_metrics.pkl")
threshold_info    = load_pkl(DIRS["thresholds"] / "threshold_info.pkl")
y_proba_tuned     = load_pkl(DIRS["reports"]    / "y_proba_tuned.pkl")
best_data         = load_pkl(DIRS["reports"]    / "best_data.pkl")
best_model_info   = load_pkl(DIRS["reports"]    / "best_model_info.pkl")
scaler            = load_pkl(DIRS["scalers"]    / "robust_scaler.pkl")
scaler_eng        = load_pkl(DIRS["scalers"]    / "robust_scaler_eng.pkl")
outlier_log       = load_pkl(DIRS["scalers"]    / "outlier_caps.pkl")
rs                = load_pkl(DIRS["reports"]    / "rs_object.pkl")
master_df         = load_pkl(DIRS["reports"]    / "master_results.pkl")

X_BEST_TR = best_data["X_BEST_TR"]
X_BEST_TE = best_data["X_BEST_TE"]
y_BEST_TR = best_data["y_BEST_TR"]
y_BEST_TE = best_data["y_BEST_TE"]

best_model_name = best_model_info["best_model_name"]
best_phase      = best_model_info["best_phase"]

t_recall90  = threshold_info["recall_90"]
best_f1_thresh   = threshold_info["optimal_f1"]
youden_thresh    = threshold_info["youden_j"]

FEATURES_ENGINEERED = load_pkl(DIRS["features"] / "features_eng.pkl")

print("✅ All Notebook 5 outputs loaded.")
print(f"  Best model  : {best_model_name}")
print(f"  Best phase  : {best_phase}")
print(f"  Deploy threshold: {t_recall90:.2f}")

✅ All Notebook 5 outputs loaded.
  Best model  : LightGBM
  Best phase  : 02_Engineered
  Deploy threshold: 0.53


### Step 16 — Learning Curve and Calibration

In [24]:
section("STEP 16 — LEARNING CURVE & CALIBRATION")

tr_sizes, tr_sc, val_sc = learning_curve(
    tuned_model, X_BEST_TR, y_BEST_TR,
    cv=cv, scoring="recall", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tr_sizes, tr_sc.mean(1),  "o-", color=COLOR_POS, lw=2.5, label="Train Recall")
ax.fill_between(tr_sizes,
                tr_sc.mean(1) - tr_sc.std(1),
                tr_sc.mean(1) + tr_sc.std(1),
                alpha=0.2, color=COLOR_POS)
ax.plot(tr_sizes, val_sc.mean(1), "o-", color=COLOR_NEG, lw=2.5, label="Validation Recall")
ax.fill_between(tr_sizes,
                val_sc.mean(1) - val_sc.std(1),
                val_sc.mean(1) + val_sc.std(1),
                alpha=0.2, color=COLOR_NEG)
ax.axhline(0.90, color=COLOR_POS, linestyle="--", lw=1.8, alpha=0.7,
           label="Medical target (Recall = 0.90)")
ax.axhspan(0.90, 1.0, alpha=0.06, color=COLOR_POS)
ax.set_title(f"Learning Curve — {best_model_name} (Tuned, scored by Recall)",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Training Set Size")
ax.set_ylabel("Recall Score")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
fig.tight_layout()
savefig(fig, DIRS["plots_best"] / "05_learning_curve.png")
plt.show()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 16 — LEARNING CURVE & CALIBRATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [PLOT]  05_learning_curve.png


In [25]:
# Skip isotonic calibration — tuned model is already well-calibrated
calibrated_model = tuned_model
t_cal_recall90   = t_recall90

print("  Isotonic calibration skipped — tuned model probabilities")
print("  are already well-scaled for this dataset.")
print(f"  Deployment model    : tuned_model  ({best_model_name})")
print(f"  Deployment threshold: {t_recall90:.2f}")

save_pkl(calibrated_model,
         DIRS["models"] / "best_model_calibrated.pkl", "Calibrated model")

  Isotonic calibration skipped — tuned model probabilities
  are already well-scaled for this dataset.
  Deployment model    : tuned_model  (LightGBM)
  Deployment threshold: 0.53
  [SAVED] Calibrated model → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\models\best_model_calibrated.pkl


### Step 17 — Final Pipeline Bundle

In [26]:
section("STEP 17 — FINAL PIPELINE BUNDLE")

if best_phase == "01_Baseline":
    FINAL_FEATURES = FEATURES_ORIGINAL
    FINAL_SCALER   = scaler
elif best_phase == "02_Engineered":
    FINAL_FEATURES = list(FEATURES_ENGINEERED)
    FINAL_SCALER   = scaler_eng
else:
    FINAL_FEATURES = top_features
    FINAL_SCALER   = scaler_eng

pipeline_bundle = {
    "model"               : tuned_model,
    "calibrated_model"    : calibrated_model,
    "scaler"              : FINAL_SCALER,
    "features_original"   : FEATURES_ORIGINAL,
    "features_engineered" : list(FEATURES_ENGINEERED),
    "features_selected"   : top_features,
    "features_used"       : list(FINAL_FEATURES),
    "target_column"       : TARGET,
    "threshold_default"   : 0.50,
    "threshold_f1"        : best_f1_thresh,
    "threshold_youden"    : youden_thresh,
    "best_model_name"     : best_model_name,
    "best_phase"          : best_phase,
    "best_params"         : rs.best_params_,
    "final_metrics"       : tuned_metrics,
    "engineer_features_fn": engineer_features,
    "outlier_caps"        : outlier_log,
}

save_pkl(pipeline_bundle, DIRS["pipeline"] / "full_pipeline_bundle.pkl", "Full pipeline bundle")

print("\n  Bundle keys saved:")
for k, v in pipeline_bundle.items():
    print(f"    {k:<30} : {type(v).__name__}")
print("\n✅ Pipeline bundle ready for Streamlit app!")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 17 — FINAL PIPELINE BUNDLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [SAVED] Full pipeline bundle → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\pipeline\full_pipeline_bundle.pkl

  Bundle keys saved:
    model                          : LGBMClassifier
    calibrated_model               : LGBMClassifier
    scaler                         : RobustScaler
    features_original              : list
    features_engineered            : list
    features_selected              : list
    features_used                  : list
    target_column                  : str
    threshold_default              : float
    threshold_f1                   : float
    threshold_youden               : float
    best_model_name                : str
    best_phase                     : str
    best_params                    : dict
    final_metrics                  : dict
   

In [27]:
# Save clean bundle (no DataFrame) for Streamlit
bundle_safe = {k: pipeline_bundle[k] for k in pipeline_bundle if k != "all_results_df"}

# ── UPDATE THIS PATH TO YOUR STREAMLIT APP FOLDER ───────────────────
streamlit_path = r"C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Application_Folder\full_pipeline_bundle.pkl"

with open(streamlit_path, "wb") as f:
    pickle.dump(bundle_safe, f)

print("✅ Safe bundle saved for Streamlit app")
print(f"   Location : {streamlit_path}")

✅ Safe bundle saved for Streamlit app
   Location : C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Application_Folder\full_pipeline_bundle.pkl


### Step 18 — Final Results and Summary Report

In [28]:
section("STEP 18 — FINAL SUMMARY REPORT")

yp_final = (y_proba_tuned >= t_recall90).astype(int)

m = dict(
    accuracy      = accuracy_score(y_BEST_TE, yp_final),
    precision     = precision_score(y_BEST_TE, yp_final, zero_division=0),
    recall        = recall_score(y_BEST_TE, yp_final, zero_division=0),
    f1            = f1_score(y_BEST_TE, yp_final, zero_division=0),
    roc_auc       = roc_auc_score(y_BEST_TE, y_proba_tuned),
    avg_precision = average_precision_score(y_BEST_TE, y_proba_tuned),
    balanced_acc  = balanced_accuracy_score(y_BEST_TE, yp_final),
    mcc           = matthews_corrcoef(y_BEST_TE, yp_final),
    cv_roc_auc    = tuned_metrics["cv_roc_auc"],
)

total_diabetics = int(y_BEST_TE.sum())
caught  = int(round(m["recall"] * total_diabetics))
missed  = total_diabetics - caught

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║            ★  FINAL RESULTS — DEPLOYMENT THRESHOLD  ★             ║
╠══════════════════════════════════════════════════════════════════════╣
║  Best Model       : {best_model_name:<49}║
║  Best Phase       : {best_phase:<49}║
║  Deploy Threshold : {t_recall90:.2f}  (Recall ≥ 90% — Medical Priority)        ║
╠══════════════════════════════════════════════════════════════════════╣
║  Recall           : {m['recall']:.4f}                                        ║
║  Precision        : {m['precision']:.4f}                                        ║
║  F1-Score         : {m['f1']:.4f}                                        ║
║  ROC-AUC          : {m['roc_auc']:.4f}                                        ║
║  PR-AUC           : {m['avg_precision']:.4f}                                        ║
║  Balanced Acc     : {m['balanced_acc']:.4f}                                        ║
║  MCC              : {m['mcc']:.4f}                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║  Caught Diabetics : {caught:,} / {total_diabetics:,}  ({caught/total_diabetics*100:.1f}%)                   ║
║  Missed Diabetics : {missed:,} / {total_diabetics:,}  ({missed/total_diabetics*100:.1f}%)                    ║
╚══════════════════════════════════════════════════════════════════════╝""")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 18 — FINAL SUMMARY REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╔══════════════════════════════════════════════════════════════════════╗
║            ★  FINAL RESULTS — DEPLOYMENT THRESHOLD  ★             ║
╠══════════════════════════════════════════════════════════════════════╣
║  Best Model       : LightGBM                                         ║
║  Best Phase       : 02_Engineered                                    ║
║  Deploy Threshold : 0.53  (Recall ≥ 90% — Medical Priority)        ║
╠══════════════════════════════════════════════════════════════════════╣
║  Recall           : 0.9064                                        ║
║  Precision        : 0.2620                                        ║
║  F1-Score         : 0.4066                                        ║
║  ROC-AUC          : 0.8186                                        ║
║  PR-AUC           : 0.4447         

In [29]:
# Final Classification Report
print("═"*65)
print(f"  FINAL CLASSIFICATION REPORT — DEPLOYMENT THRESHOLD")
print(f"  Model     : {best_model_name}")
print(f"  Threshold : {t_recall90:.2f}  (Recall ≥ 90% — Medical Priority)")
print("═"*65)
print(classification_report(
    y_BEST_TE, yp_final,
    target_names=["No Diabetes (0)", "Diabetes (1)"],
    digits=4))
print("═"*65)
print(f"  Caught Diabetics : {caught:,} / {total_diabetics:,}  ({caught/total_diabetics*100:.1f}%)")
print(f"  Missed Diabetics : {missed:,} / {total_diabetics:,}  ({missed/total_diabetics*100:.1f}%)")
print("═"*65)

═════════════════════════════════════════════════════════════════
  FINAL CLASSIFICATION REPORT — DEPLOYMENT THRESHOLD
  Model     : LightGBM
  Threshold : 0.53  (Recall ≥ 90% — Medical Priority)
═════════════════════════════════════════════════════════════════
                 precision    recall  f1-score   support

No Diabetes (0)     0.9696    0.5392    0.6930     38876
   Diabetes (1)     0.2620    0.9064    0.4066      7019

       accuracy                         0.5953     45895
      macro avg     0.6158    0.7228    0.5498     45895
   weighted avg     0.8614    0.5953    0.6492     45895

═════════════════════════════════════════════════════════════════
  Caught Diabetics : 6,362 / 7,019  (90.6%)
  Missed Diabetics : 657 / 7,019  (9.4%)
═════════════════════════════════════════════════════════════════


### Research-Quality Final Plots

In [30]:
RESEARCH_DIR = OUTPUT_ROOT / "research_plots"
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

# PLOT 1 — Final Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_BEST_TE, yp_final)
tn, fp, fn, tp = cm.ravel()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes","Diabetes"],
            yticklabels=["No Diabetes","Diabetes"],
            linewidths=2, linecolor="white", ax=ax,
            annot_kws={"size": 16, "weight": "bold"})
rec  = recall_score(y_BEST_TE, yp_final, zero_division=0)
prec = precision_score(y_BEST_TE, yp_final, zero_division=0)
f1v  = f1_score(y_BEST_TE, yp_final, zero_division=0)
ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold")
ax.set_ylabel("True Label",      fontsize=13, fontweight="bold")
ax.set_title(f"Confusion Matrix — {best_model_name}\n"
             f"Threshold={t_recall90:.2f} | Recall={rec:.4f} | "
             f"Precision={prec:.4f} | F1={f1v:.4f}",
             fontsize=11, fontweight="bold")
fig.tight_layout()
fig.savefig(RESEARCH_DIR / "01_confusion_matrix_final.png", dpi=200, bbox_inches="tight")
plt.show()
print("[SAVED] 01_confusion_matrix_final.png")

[SAVED] 01_confusion_matrix_final.png


In [31]:
# PLOT 2 — Final ROC Curve
fig, ax = plt.subplots(figsize=(7, 6))
fpr_r, tpr_r, _ = roc_curve(y_BEST_TE, y_proba_tuned)
auc_val = roc_auc_score(y_BEST_TE, y_proba_tuned)
ax.plot(fpr_r, tpr_r, color=COLOR_ROC, lw=2.5, label=f"LightGBM  AUC={auc_val:.4f}")
ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.5)
yp_deploy = (y_proba_tuned >= t_recall90).astype(int)
fpr_pt = (yp_deploy[y_BEST_TE == 0] == 1).mean()
tpr_pt = recall_score(y_BEST_TE, yp_deploy, zero_division=0)
ax.scatter([fpr_pt], [tpr_pt], s=150, color=COLOR_POS, zorder=5,
           label=f"Recall≥90% (t={t_recall90:.2f}) ★")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title=f"ROC Curve — {best_model_name}")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESEARCH_DIR / "02_roc_curve_final.png", dpi=200, bbox_inches="tight")
plt.show()
print("[SAVED] 02_roc_curve_final.png")

[SAVED] 02_roc_curve_final.png


In [32]:
# PLOT 3 — Final PR Curve
fig, ax = plt.subplots(figsize=(7, 6))
prec_c, rec_c, _ = precision_recall_curve(y_BEST_TE, y_proba_tuned)
ap = average_precision_score(y_BEST_TE, y_proba_tuned)
baseline_ap = y_BEST_TE.mean()
ax.plot(rec_c, prec_c, color=COLOR_PR, lw=2.5, label=f"LightGBM  AP={ap:.4f}")
ax.axhline(baseline_ap, color="gray", linestyle="--",
           label=f"Baseline AP={baseline_ap:.4f}")
ax.set(xlabel="Recall", ylabel="Precision",
       title=f"Precision-Recall Curve — {best_model_name}")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESEARCH_DIR / "03_pr_curve_final.png", dpi=200, bbox_inches="tight")
plt.show()
print("[SAVED] 03_pr_curve_final.png")

[SAVED] 03_pr_curve_final.png


In [33]:
# PLOT 4 — Feature Importance
rf_imp    = load_pkl(DIRS["features"] / "rf_importance.pkl")
mi_series = load_pkl(DIRS["features"] / "mi_scores.pkl")
combined  = load_pkl(DIRS["features"] / "combined_rank.pkl")

fig, ax = plt.subplots(figsize=(10, 7))
top_combined = combined.sort_values().head(TOP_N)
ax.barh(top_combined.index[::-1], top_combined.values[::-1],
        color=PALETTE[1], edgecolor="white")
ax.set_title(f"Top {TOP_N} Features — Combined MDI + MI Rank",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Combined Rank Score (lower = more important)")
ax.invert_xaxis()
fig.tight_layout()
fig.savefig(RESEARCH_DIR / "07_feature_importance_final.png", dpi=200, bbox_inches="tight")
plt.show()
print("[SAVED] 07_feature_importance_final.png")

[SAVED] 07_feature_importance_final.png


In [34]:
# Save final summary JSON
summary = {
    "project"      : "Smart Diabetes Monitoring and Prediction System",
    "dataset_type" : "UNBALANCED",
    "team"         : ["Ahmad Azhar Almansoor", "Ibrahim Madian Fadhil"],
    "timestamp"    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_model"   : best_model_name,
    "best_phase"   : best_phase,
    "best_params"  : {k: str(v) for k, v in rs.best_params_.items()},
    "thresholds"   : threshold_info,
    "final_metrics": {k: round(v, 4) for k, v in m.items()},
    "caught"       : caught,
    "missed"       : missed,
    "total_diabetics": total_diabetics,
}

with open(DIRS["reports"] / "final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
save_pkl(summary, DIRS["reports"] / "final_summary.pkl", "Final summary")

print("\n✅ All research plots and summary saved.")
print(f"   Research plots → {RESEARCH_DIR.resolve()}")

  [SAVED] Final summary → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\final_summary.pkl

✅ All research plots and summary saved.
   Research plots → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\research_plots


### ✅ Project Complete

All 6 notebooks have been run successfully. Your deployment bundle is at:
`diabetes_unbalanced_outputs/pipeline/full_pipeline_bundle.pkl`

And the Streamlit-ready safe bundle has been saved to your app folder.